# 학습 파이프라인 구축

Pytorch를 사용하여 학습 파이프라인을 구축하는 과제 노트북입니다.

진행 순서는 다음과 같습니다.

1. 데이터 다운로드 후 전처리
2. Pytorch Dataset 생성
3. Pytorch DataLoader 생성
4. 모델 구축
5. 모델 학습

## 1. 데이터를 다운로드 후 전처리
    - 예측 값 y는 성별(sex)와 생존여부(survived)이며 입력 데이터 X에 포함되어선 안 됩니다.
    - 입력할 데이터 X에 포함될 수치형 데이터는 평균이 0, 분산이 1인 표준편차가 되어야합니다.
    - 입력할 데이터 X에 범주형 데이터를 사용할 경우 class로 변환되어야합니다.
    - Train 데이터와 Test 데이터를 8:2로 분할하고 만일 Validation 데이터까지 포함한다면 8:1:1 비율로 나누십시오.

### 데이터 다운로드

In [19]:
import numpy as np
import pandas as pd
import seaborn as sns

In [66]:
df = sns.load_dataset("titanic")
print(df.shape)
df.info()

(891, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


### 정보 누설 방지 및 데이터 전처리

In [65]:
y = df[['sex', 'survived']].copy()
x = df.drop(columns=['sex', 'survived'])
x = x.drop(columns=['alive', 'who','adult_male','deck'])
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   pclass       891 non-null    int64   
 1   age          714 non-null    float64 
 2   sibsp        891 non-null    int64   
 3   parch        891 non-null    int64   
 4   fare         891 non-null    float64 
 5   embarked     889 non-null    object  
 6   class        891 non-null    category
 7   embark_town  889 non-null    object  
 8   alone        891 non-null    bool    
dtypes: bool(1), category(1), float64(2), int64(3), object(2)
memory usage: 50.7+ KB


### feature 선택

In [54]:
# 중복 또는 파생 feature 제거
drop_features = ['class', 'embark_town', 'alone']
numeric_features = ['pclass', 'age', 'sibsp', 'parch', 'fare']

x = x.drop(columns=drop_features)

# X의 범주형 feature 결측치 처리
x['embarked'] = x['embarked'].fillna(x['embarked'].mode()[0])
# X의 수치형 feature 결측치 처리
x['fare'] = x['fare'].fillna(x['fare'].mean())
x['age'] = x['age'].fillna(x['age'].median())

# X의 범주형 feature class 변환
embarked_class = {
    'C': 0,
    'Q': 1,
    'S': 2
}

x['embarked'] = x['embarked'].map(embarked_class)

# y의 범주형 target class 변환
sex_class = {
    'male': 0,
    'female': 1
}

y['sex'] = y['sex'].map(sex_class)


### train / vaildation / test 데이터 분리

In [55]:
from sklearn.model_selection import train_test_split

# y가 sex, survived 두 개이므로 두 target의 조합을 층화 기준으로 사용
stratify_label = y['sex'].astype(str) + "_" + y['survived'].astype(str)

# 1차 분리: train 80%, temp 20%
x_train, x_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_label
)

# temp 데이터에 대한 stratify 기준 다시 생성
stratify_temp = y_temp['sex'].astype(str) + "_" + y_temp['survived'].astype(str)

# 2차 분리: temp 20%를 validation 10%, test 10%로 분리
x_valid, x_test, y_valid, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=stratify_temp
)

# 분리 결과 확인
print("x_train:", x_train.shape)
print("x_valid:", x_valid.shape)
print("x_test :", x_test.shape)

print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)
print("y_test :", y_test.shape)

x_train: (712, 6)
x_valid: (89, 6)
x_test : (90, 6)
y_train: (712, 2)
y_valid: (89, 2)
y_test : (90, 2)


In [56]:
# feature 관계성 확인용 데이터 생성
relation_df = x_train.copy()
relation_df['sex'] = y_train['sex']
relation_df['survived'] = y_train['survived']

# 수치형 feature와 target 간 상관관계 확인
target_corr = relation_df[numeric_features + ['embarked', 'sex', 'survived']].corr()
print(target_corr[['sex', 'survived']])
# feature와 target 간 상관관계 계산
corr_with_targets = relation_df[numeric_features + ['embarked', 'sex', 'survived']].corr()[['sex', 'survived']]

# target 자기 자신은 제외
corr_with_targets = corr_with_targets.drop(index=['sex', 'survived'])

# 두 target 중 더 큰 절댓값 기준으로 관련성 점수 계산
feature_scores = corr_with_targets.abs().max(axis=1)
print(feature_scores.sort_values(ascending=False))
selected_features = feature_scores[feature_scores >= 0.05].index.tolist()

print("Selected Features:", selected_features)

               sex  survived
pclass   -0.114383 -0.300005
age      -0.099928 -0.063049
sibsp     0.117989 -0.043690
parch     0.254768  0.070295
fare      0.180629  0.238073
embarked -0.089315 -0.156579
sex       1.000000  0.542702
survived  0.542702  1.000000
pclass      0.300005
parch       0.254768
fare        0.238073
embarked    0.156579
sibsp       0.117989
age         0.099928
dtype: float64
Selected Features: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'embarked']


In [57]:
#선택된 feature만 남기
x_train = x_train[selected_features]
x_valid = x_valid[selected_features]
x_test = x_test[selected_features]

numeric_features = [col for col in numeric_features if col in selected_features]

In [58]:
#표준화 작업
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train[numeric_features] = scaler.fit_transform(x_train[numeric_features])
x_valid[numeric_features] = scaler.transform(x_valid[numeric_features])
x_test[numeric_features] = scaler.transform(x_test[numeric_features])

## 2. Pytorch Dataset 생성


In [63]:
import torch
from torch.utils.data import Dataset, DataLoader
class TitanicDataset(Dataset):
    def __init__(self, x, y):
      self.x = torch.tensor(x.values, dtype=torch.float32)
      self.y = torch.tensor(y.values, dtype=torch.float32)
      self.sex = torch.tensor(
          y["sex"].values,
          dtype=torch.float32
      ).unsqueeze(1)
      self.survived = torch.tensor(
          y["survived"].values,
          dtype=torch.float32
      ).unsqueeze(1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "inputs": self.x[idx],
            "sex": self.sex[idx],
            "survived": self.survived[idx]
            }
train_ds = TitanicDataset(x_train, y_train)
val_ds   = TitanicDataset(x_valid, y_valid)
test_ds  = TitanicDataset(x_test, y_test)

# 출력 형식 확인
print(train_ds[0])

{'inputs': tensor([ 0.8169, -0.0791,  0.4621, -0.4761, -0.3224,  1.0000]), 'sex': tensor([1.]), 'survived': tensor([1.])}


## 3. Pytorch DataLoader 생성

In [64]:
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds,batch_size = BATCH_SIZE,shuffle=False)

batch = next(iter(train_loader))
print("input:", batch["inputs"].shape)
print("label:", batch["sex"].shape, batch["survived"].shape)

input: torch.Size([32, 6])
label: torch.Size([32, 1]) torch.Size([32, 1])


## 4. 모델 구축

In [68]:
import torch.nn as nn
class MLP(nn.Module):
  def __init__(self, in_feature = 6):
     super().__init__()
     self.shared = nn.Sequential(
         nn.Linear(in_feature, 32, bias=True),
         nn.ReLU(),
         nn.Linear(in_feature, 32, bias=True),
         nn.ReLU()
     )
     self.sex_head = nn.Sequential(
         nn.Linear(16, 8),
         nn.ReLU(),
         nn.Linear(8, 1),
            nn.Sigmoid(),
         )
     self.sur_head = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid(),
            )


  def forward(self, x):
    h = self.shared(x)
    return self.sex_head(h), self.sur_head(h)
model = MLP(in_feature = x_train.shape[1])
model


MLP(
  (shared): Sequential(
    (0): Linear(in_features=6, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=6, out_features=32, bias=True)
    (3): ReLU()
  )
  (sex_head): Sequential(
    (0): Linear(in_features=16, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
    (3): Sigmoid()
  )
  (sur_head): Sequential(
    (0): Linear(in_features=16, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
    (3): Sigmoid()
  )
)

## 5. 모델 학습

### Loss function과 optimizer 설정

In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [70]:
criterion = nn.BCELoss()  # Sigmoid 출력이라 BCELoss 사용
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 50

### 학습 루프 함수 작성

In [71]:
from tqdm.notebook import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0.0
    correct_sex = 0
    correct_survived = 0
    total_count = 0

    pbar = tqdm(
        loader,
        desc="Train",
        leave=False
    )

    for batch in pbar:

        X = batch["inputs"].to(device)

        y_sex = batch["sex"].to(device)
        y_survived = batch["survived"].to(device)

        pred_sex, pred_survived = model(X)

        loss = (
            criterion(pred_sex, y_sex) +
            criterion(pred_survived, y_survived)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        correct_sex += (
            (pred_sex > 0.5) == y_sex
        ).sum().item()

        correct_survived += (
            (pred_survived > 0.5) == y_survived
        ).sum().item()

        total_count += X.size(0)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return {
        "loss": total_loss / total_count,
        "sex_acc": correct_sex / total_count,
        "survived_acc": correct_survived / total_count,
    }

### 모델 학습 실행

In [ ]:
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0
    correct_sex = 0
    correct_survived = 0
    total_count = 0

    pbar = tqdm(
        loader,
        desc="Eval",
        leave=False
    )

    with torch.no_grad():

        for batch in pbar:

            X = batch["inputs"].to(device)

            y_sex = batch["sex"].to(device)
            y_survived = batch["survived"].to(device)

            pred_sex, pred_survived = model(X)

            loss = (
                criterion(pred_sex, y_sex) +
                criterion(pred_survived, y_survived)
            )

            total_loss += loss.item() * X.size(0)

            correct_sex += (
                (pred_sex > 0.5) == y_sex
            ).sum().item()

            correct_survived += (
                (pred_survived > 0.5) == y_survived
            ).sum().item()

            total_count += X.size(0)

            pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

    return {
        "loss": total_loss / total_count,
        "sex_acc": correct_sex / total_count,
        "survived_acc": correct_survived / total_count,
    }